In [ ]:
!pip install darts -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.7/204.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 51.4 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
from pathlib import Path

from darts import TimeSeries
from sklearn.preprocessing import MinMaxScaler
from darts.models import RNNModel
from darts.metrics import rmse, mae, mape
from pytorch_lightning.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import torch
torch.set_float32_matmul_precision("high")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- CONFIG ---
RESULTS_PATH = "/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_log/gru_1layer_subnets_global_rolling_metrics.csv"

# Load Data
df = pd.read_csv(RESULTS_PATH)

# Basic Stats for the Text
median_r2 = df['R2_LogScaled'].median()
mean_rmse = df['RMSE_LogScaled'].mean()


success_count = len(df[df['R2_LogScaled'] > 0])
total_count = len(df)
success_rate = (success_count / total_count) * 100

# High Accuracy = R2 > 0.5
high_accuracy = len(df[df['R2_LogScaled'] > 0.5]) / total_count * 100

print(f"=== RESULTS FOR TEXT SECTION 4.1 ===")
print(f"Total Subnets: {total_count}")
print(f"Median R2: {median_r2:.4f}  <-- Put this in text")
print(f"Success Rate (R2 > 0): {success_rate:.1f}%  <-- Put this in text")
print(f"High Accuracy (R2 > 0.5): {high_accuracy:.1f}%  <-- Put this in text")

# --- PLOT ---
plt.figure(figsize=(10, 6))


plot_data = df['R2_LogScaled'].clip(lower=-1.0)

sns.histplot(plot_data, bins=30, kde=True, color='#2980b9', edgecolor='black')

plt.axvline(x=median_r2, color='green', linestyle='--', linewidth=2, label=f"Median $R^2$: {median_r2:.2f}")
plt.axvline(x=0, color='red', linestyle='-', linewidth=2, label="Baseline (Mean)")

plt.title("Distribution of Global GRU Performance (541 Subnets)", fontsize=14, fontweight='bold')
plt.xlabel("Coefficient of Determination ($R^2$)", fontsize=12)
plt.ylabel("Frequency (Number of Subnets)", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("global_gru_histogram.png")
plt.show()
print("Saved plot as 'global_gru_histogram.png'")

In [ ]:
DATA_DIR   = Path("/content/drive/MyDrive/Thesis/Datasets/network_anomaly")
SUBNET_DIR = DATA_DIR / "institution_subnets/agg_1_hour"

TIMES_PATH = DATA_DIR / "times/times_1_hour.csv"
FEATURE = "n_bytes"
FEATURE_COLS = [FEATURE]

times_df = pd.read_csv(TIMES_PATH)

times_df["time"] = pd.to_datetime(times_df["time"], errors="coerce")
if times_df["time"].dt.tz is not None:
    times_df["time"] = times_df["time"].dt.tz_localize(None)

times_df["id_time"] = times_df["id_time"].astype(int)

from darts import TimeSeries

raw_series = {}
ts_dict    = {}

for csv_path in SUBNET_DIR.glob("*.csv"):
    sid = csv_path.stem
    df = pd.read_csv(csv_path)

    df["id_time"] = df["id_time"].astype(int)

    df = df.merge(times_df, on="id_time", how="left")

    df = df.sort_values("time")
    df = df.set_index("time")

    # keep only n_bytes
    df = df[FEATURE_COLS].copy()

    raw_series[sid] = df

    ts = TimeSeries.from_dataframe(
        df,
        value_cols=FEATURE_COLS,
        fill_missing_dates=True,
        freq="h"
    )
    ts_dict[sid] = ts

print(f"Loaded {len(ts_dict)} subnets into ts_dict")



In [ ]:
from darts.dataprocessing.transformers import MissingValuesFiller
import numpy as np

filler = MissingValuesFiller()

filled_ts_dict = {}

for sid, ts in ts_dict.items():
    ts_filled = filler.transform(ts)
    n_nans = np.isnan(ts_filled.values()).sum()
    print(f"{sid}: NaNs AFTER filling = {n_nans}")
    filled_ts_dict[sid] = ts_filled


In [ ]:
train_ts_dict = {}
val_ts_dict   = {}
test_ts_dict  = {}

for sid, ts in filled_ts_dict.items():
    n = len(ts)
    train_end = int(n * 0.35)
    val_end   = int(n * 0.40)  # 35% + 5%

    train_ts_dict[sid] = ts[:train_end]
    val_ts_dict[sid]   = ts[train_end:val_end]
    test_ts_dict[sid]  = ts[val_end:]


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

train_scaled_log = {}
val_scaled_log   = {}
test_scaled_log  = {}
scalers_log      = {}   # per-subnet scaler in log-space

for sid in train_ts_dict.keys():
    train_ts = train_ts_dict[sid]
    val_ts   = val_ts_dict[sid]
    test_ts  = test_ts_dict[sid]

    # raw values (n_bytes)
    train_vals = train_ts.values()   # (T_train, 1)
    val_vals   = val_ts.values()
    test_vals  = test_ts.values()

    # ---- log1p transform ----
    train_log = np.log1p(train_vals)
    val_log   = np.log1p(val_vals)
    test_log  = np.log1p(test_vals)

    # concatenate train+val in log-space
    tv_log = np.concatenate([train_log, val_log], axis=0)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(tv_log)

    train_scaled_vals = scaler.transform(train_log)
    val_scaled_vals   = scaler.transform(val_log)
    test_scaled_vals  = scaler.transform(test_log)

    # wrap back into TimeSeries
    train_scaled_log[sid] = train_ts.with_values(train_scaled_vals)
    val_scaled_log[sid]   = val_ts.with_values(val_scaled_vals)
    test_scaled_log[sid]  = test_ts.with_values(test_scaled_vals)

    scalers_log[sid] = scaler


train_scaled = {}
val_scaled   = {}
test_scaled  = {}
scalers      = {}   # subnet_id -> its own MinMaxScaler

for sid in train_ts_dict.keys():
    train_ts = train_ts_dict[sid]
    val_ts   = val_ts_dict[sid]
    test_ts  = test_ts_dict[sid]

    # concatenate train+val
    tv_vals = np.concatenate([train_ts.values(), val_ts.values()], axis=0)  # shape (N,1)

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(tv_vals)

    # scale each split separately using the SAME scaler
    train_scaled_vals = scaler.transform(train_ts.values())
    val_scaled_vals   = scaler.transform(val_ts.values())
    test_scaled_vals  = scaler.transform(test_ts.values())

    train_scaled[sid] = train_ts.with_values(train_scaled_vals)
    val_scaled[sid]   = val_ts.with_values(val_scaled_vals)
    test_scaled[sid]  = test_ts.with_values(test_scaled_vals)

    scalers[sid] = scaler
valid_ids = []

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- CONFIG ---
SAVE_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Methodology_Plots")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Pick a representative subnet (e.g., Subnet 9)
target_sid = '9'

if target_sid in train_scaled_log:
    # 1. Get the actual series from your dictionaries
    train_ts = train_scaled_log[target_sid].to_series()
    val_ts   = val_scaled_log[target_sid].to_series()
    test_ts  = test_scaled_log[target_sid].to_series()

    # 2. Plotting
    plt.figure(figsize=(12, 5))

    # Plot Train (Blue)
    plt.plot(train_ts.index, train_ts.values, label="Training (35%)", color='#2c3e50')

    # Plot Validation (Orange)
    plt.plot(val_ts.index, val_ts.values, label="Validation (5%)", color='#e67e22')

    # Plot Test (Green)
    plt.plot(test_ts.index, test_ts.values, label="Test (60%)", color='#27ae60')

    # Add vertical split lines
    plt.axvline(x=val_ts.index[0], color='black', linestyle='--', alpha=0.5)
    plt.axvline(x=test_ts.index[0], color='black', linestyle='--', alpha=0.5)

    plt.title(f"Data Splitting Strategy (Subnet {target_sid})", fontsize=14, fontweight='bold')
    plt.ylabel("Log Volume", fontsize=12)
    plt.legend(loc="upper right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig(SAVE_DIR / "data_splitting_visual_actual.png")
    plt.show()
    print("Split visualization saved.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from darts import TimeSeries

# --- CONFIG ---
TARGET_SID = '28'  # The noisy subnet

# Check if data exists in your dictionary
if TARGET_SID in train_scaled_log:
    print(f"Plotting Deep Inspection for Subnet {TARGET_SID} (Log Scale)...")

    # Get the Data
    train_series = train_scaled_log[TARGET_SID]
    test_series  = test_scaled_log[TARGET_SID]

    # PLOT
    plt.figure(figsize=(12, 6))

    # Plot Training Data (Blue)
    train_series.plot(label="Training Data (Learned)", color='#2980b9')

    # Plot Test Data (Green)
    test_series.plot(label="Test Data (Failed)", color='#27ae60')

    split_time = train_series.end_time()
    plt.axvline(x=split_time, color='black', linestyle='--', label="Train/Test Split", alpha=0.7)


    plt.title(f"Visualizing The 'Overfitting Trap': Subnet {TARGET_SID}", fontsize=14, fontweight='bold')
    plt.ylabel("Log Volume (n_bytes)")
    plt.xlabel("Time")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.show()

    # Get raw values as numpy arrays
    train_vals = train_series.values().flatten()
    test_vals  = test_series.values().flatten()
    full_vals  = np.concatenate([train_vals, test_vals])

    std_dev = np.std(full_vals)
    mean_val = np.mean(full_vals)


    if mean_val == 0:
        cv = 0
    else:
        cv = std_dev / mean_val

    print(f"\n--- Statistics for Subnet {TARGET_SID} ---")
    print(f"Standard Deviation: {std_dev:.4f}")
    print(f"Mean Value: {mean_val:.4f}")
    print(f"Coefficient of Variation (Volatility): {cv:.4f}")

    if cv > 0.5:
        print(">>High Volatility. The data is noisy/jagged.")
        print("   This explains why the 3-Layer model overfit (Negative R2).")
        print("   It tried to memorize the jagged spikes.")
    else:
        print(">>Low/Moderate Volatility.")

else:
    print(f"Subnet {TARGET_SID} not found in the loaded dataset.")

In [ ]:
from darts.models import RNNModel
from darts.metrics import rmse, r2_score
from pytorch_lightning.callbacks import EarlyStopping
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- CONFIG ---
MODEL_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/LSTM_168_1layer_RAW")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = MODEL_DIR / "plots_rolling_check"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Define Early Stopping
early_stop_lstm = EarlyStopping(
    monitor="val_loss",
    patience=7,
    min_delta=1e-4,
    mode="min",
)

# --- STEP 0: POPULATE valid_ids ---
INPUT_CHUNK_LENGTH = 168
MIN_LEN = INPUT_CHUNK_LENGTH + 1

valid_ids = []

print("Filtering valid subnets (RAW)...")

# Iterate through your loaded data to find subnets with enough history
for sid, ts in train_scaled.items():
    len_train = len(ts)
    len_val   = len(val_scaled[sid])

    if len_train >= MIN_LEN and len_val >= MIN_LEN:
        valid_ids.append(sid)

print(f"✅ Found {len(valid_ids)} valid subnets.")

# update the lists for training
train_scaled_list = [train_scaled[sid] for sid in valid_ids]
val_scaled_list   = [val_scaled[sid]   for sid in valid_ids]

# --- 1. DEFINE & TRAIN MODEL ---
lstm_global = RNNModel(
    model="LSTM",
    input_chunk_length=168,
    training_length=168,
    hidden_dim=64,
    n_rnn_layers=1,          # <--- 1 or 3 Layers, adjust accordingly
    dropout=0.1,
    batch_size=64,
    n_epochs=30,
    optimizer_kwargs={"lr": 5e-4},
    random_state=42,
    model_name="lstm_subnets_168_1layer_global_raw",
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "devices": 1,
        "gradient_clip_val": 1.0,
        "callbacks": [early_stop_lstm],
    },
)

print(f"Training Global 1-Layer LSTM (RAW) on {len(valid_ids)} subnets...")

lstm_global.fit(
    series=train_scaled_list,
    val_series=val_scaled_list,
    verbose=True,
)

# SAVE THE MODEL
save_path = MODEL_DIR / "lstm_168_1layer_subnets_raw.pth.tar"
lstm_global.save(str(save_path))
print(f"Model saved to: {save_path}")


# --- 2. ROLLING EVALUATION ---
print(f"\nStarting ROLLING evaluation on {len(valid_ids)} subnets...")
results_global = []

for i, sid in enumerate(valid_ids):

    # Get raw Data
    train_ts = train_scaled[sid]
    val_ts   = val_scaled[sid]
    test_ts  = test_scaled[sid]

    # Concatenate full history
    full_series = train_ts.concatenate(val_ts).concatenate(test_ts)

    try:
        # ROLLING FORECAST
        pred = lstm_global.historical_forecasts(
            series=full_series,
            start=test_ts.start_time(),
            forecast_horizon=1,
            stride=1,
            retrain=False,
            verbose=False,
            last_points_only=True
        )

        # METRICS (On Raw Scaled Data)
        r2_val = r2_score(test_ts, pred)
        rmse_val = rmse(test_ts, pred)

        results_global.append({
            "subnet": sid,
            "R2_Scaled": r2_val,
            "RMSE_Scaled": rmse_val,
            "n_test_points": len(test_ts)
        })

        # PLOTTING (First 3 subnets)
        if i < 3:
            plt.figure(figsize=(12, 5))
            plt.plot(test_ts.time_index, test_ts.values(), label="Actual (Raw)", color='blue', alpha=0.5)
            plt.plot(pred.time_index, pred.values(), label="Global 1L Pred (Raw)", color='green', alpha=0.8)
            plt.title(f"Global 1-Layer (Raw) | Subnet {sid} | R2: {r2_val:.4f}")
            plt.legend()
            plt.grid(True, alpha=0.3)

            p_file = PLOT_DIR / f"global_1layer_raw_{sid}.png"
            plt.savefig(p_file)
            plt.close()
            print(f"   [{i+1}] Subnet {sid}: R2={r2_val:.4f} -> Plot saved")

    except Exception as e:
        print(f"Error evaluating subnet {sid}: {e}")

#Saving
if results_global:
    df_results = pd.DataFrame(results_global)

    print("\n=== Global 1-Layer Subnet (Raw) Results ===")
    print(df_results[["R2_Scaled", "RMSE_Scaled"]].mean())

    csv_path = MODEL_DIR / "lstm_1layer_subnets_rolling_metrics.csv"
    df_results.to_csv(csv_path, index=False)
    print(f"\n✅ Results saved to: {csv_path}")

In [ ]:

from darts.models import RNNModel
from darts.metrics import rmse, r2_score
from pytorch_lightning.callbacks import EarlyStopping
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- CONFIG ---
MODEL_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_raw")
PLOT_DIR = MODEL_DIR / "plots_rolling_check"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CHUNK = 168
MIN_LEN = INPUT_CHUNK + 1

# --- 1. FILTER & PREPARE DATA (LOG SCALED) ---
# We use Log data because it usually handles network spikes better
print("Filtering valid subnets for GRU Training...")
valid_subnet_ids = []
train_list = []
val_list = []

for sid, ts in train_scaled.items():
    sid_str = str(sid)
    # Check lengths
    if len(ts) >= MIN_LEN and len(val_scaled[sid_str]) >= MIN_LEN:
        valid_subnet_ids.append(sid_str)
        train_list.append(train_scaled[sid_str])
        val_list.append(val_scaled[sid_str])

print(f"✅ Found {len(valid_subnet_ids)} valid subnets. Starting GRU training...")

# --- 2. TRAIN MODEL (GRU) ---
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    min_delta=1e-4,
    mode="min",
)

gru_global = RNNModel(
    model="GRU",
    input_chunk_length=INPUT_CHUNK,
    training_length=INPUT_CHUNK,
    hidden_dim=64,
    n_rnn_layers=1,
    dropout=0.1,
    batch_size=64,
    n_epochs=20,
    optimizer_kwargs={"lr": 1e-3},
    random_state=42,
    model_name="gru_subnets_168_1layer_raw",
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "devices": 1,
        "callbacks": [early_stop],
    },
)

gru_global.fit(
    series=train_list,
    val_series=val_list,
    verbose=True,
)

# Save the trained model
save_path = MODEL_DIR / "gru_168_1layer_subnets.pth.tar"
gru_global.save(str(save_path))
print(f"GRU Model sved to: {save_path}")


# --- 3. ROLLING EVALUATION (GRU) ---
print(f"\nStarting GRU ROLLING evaluation on {len(valid_subnet_ids)} subnets...")
results_global = []

for i, sid in enumerate(valid_subnet_ids):

    # Get LOG Scaled Data
    train_ts = train_scaled[sid]
    val_ts   = val_scaled[sid]
    test_ts  = test_scaled[sid]

    full_series = train_ts.concatenate(val_ts).concatenate(test_ts)

    try:
        # ROLLING FORECAST
        pred = gru_global.historical_forecasts(
            series=full_series,
            start=test_ts.start_time(),
            forecast_horizon=1,
            stride=1,
            retrain=False,
            verbose=False,
            last_points_only=True
        )

        # METRICS (On Log Scaled Data)
        r2_val = r2_score(test_ts, pred)
        rmse_val = rmse(test_ts, pred)

        results_global.append({
            "subnet": sid,
            "R2_Scaled": r2_val,
            "RMSE_Scaled": rmse_val,
            "n_test_points": len(test_ts)
        })

        # Plot first 3 subnets
        if i < 3:
            plt.figure(figsize=(12, 5))
            plt.plot(test_ts.time_index, test_ts.values(), label="Actual (Raw)", color='blue', alpha=0.5)
            plt.plot(pred.time_index, pred.values(), label="Global GRU Pred", color='magenta', alpha=0.8)
            plt.title(f"Global GRU (1L) | Subnet {sid} | R2: {r2_val:.4f}")
            plt.legend()
            plt.grid(True, alpha=0.3)

            p_path = PLOT_DIR / f"gru_global_subnet_{sid}.png"
            plt.savefig(p_path)
            plt.close()
            print(f"   [{i+1}] Subnet {sid}: R2={r2_val:.4f} -> Plot saved")

    except Exception as e:
        print(f"Error on subnet {sid}: {e}")

# --- 4. SAVE RESULTS ---
if results_global:
    df_global = pd.DataFrame(results_global)
    print("\n=== Global GRU Subnet Results ===")
    print(df_global[["R2_Scaled", "RMSE_Scaled"]].mean())

    csv_path = MODEL_DIR / "gru_1layer_subnets_global_rolling_metrics.csv"
    df_global.to_csv(csv_path, index=False)
    print(f"\n Saved GRU results to: {csv_path}")

In [ ]:
valid_subnet_ids = []
train_list = []
val_list = []

INPUT_CHUNK = 168
MIN_LEN = INPUT_CHUNK + 1

for sid, ts in train_scaled.items():
    sid_str = str(sid)
    # Check lengths
    if len(ts) >= MIN_LEN and len(val_scaled[sid_str]) >= MIN_LEN:
        valid_subnet_ids.append(sid_str)
        train_list.append(train_scaled[sid_str])
        val_list.append(val_scaled[sid_str])

print(f"Found {len(valid_subnet_ids)} valid subnets. Starting GRU training...")

In [ ]:
from darts.models import RNNModel
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- CONFIG ---
MODEL_PATH = Path("/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_log/gru_168_1layer_subnets_log.pth.tar")
ANOMALY_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Error_Distribution")
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)

K_SIGMA = 3.0
WINDOW_SIZE = 24

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150

# --- LOAD MODEL ---
print(f"Loading Champion Model...")
model = RNNModel.load(str(MODEL_PATH))
print("✅ Model loaded!")

# --- TARGET SELECTION ---
targets = ['0']
print(f"Generating distribution plots for subnets: {targets}...")

for sid in targets:
    sid_str = str(sid)

    # 1. Get Data
    if sid_str not in train_scaled_log:
        print(f"Skipping {sid_str} (not in data)")
        continue

    train_ts = train_scaled_log[sid_str]
    val_ts   = val_scaled_log[sid_str]
    test_ts  = test_scaled_log[sid_str]
    full_series = train_ts.concatenate(val_ts).concatenate(test_ts)

    # 2. Predict (Historical Forecast)
    print(f"Processing Subnet {sid_str}...")
    pred = model.historical_forecasts(
        series=full_series,
        start=test_ts.start_time(),
        forecast_horizon=1,
        stride=1,
        retrain=False,
        verbose=False,
        last_points_only=True
    )

    # 3. Calculate Error
    intersect_idx = test_ts.time_index.intersection(pred.time_index)
    actual_aligned = test_ts.slice_intersect(pred)
    pred_aligned = pred.slice_intersect(test_ts)

    # Compute Absolute Error
    error_series = (actual_aligned - pred_aligned).map(np.abs)
    error_values = error_series.values().flatten()

    # 4. Dynamic Threshold Stats
    error_pd = pd.Series(error_values, index=error_series.time_index)
    rolling_mean = error_pd.rolling(window=WINDOW_SIZE, min_periods=1).mean()
    rolling_std  = error_pd.rolling(window=WINDOW_SIZE, min_periods=1).std()
    threshold = rolling_mean + (K_SIGMA * rolling_std)

    # --- NEW: ERROR DISTRIBUTION PLOT ---
    plt.figure(figsize=(10, 6))

    # A. Histogram with KDE
    sns.histplot(error_pd.values, bins=50, kde=True, color='#2c3e50', alpha=0.6, label='Error Frequency')

    # B. Add Mean Threshold Line
    avg_thresh = threshold.mean()
    plt.axvline(x=avg_thresh, color='#c0392b', linestyle='--', linewidth=2, label=f"Avg Threshold (~{avg_thresh:.2f})")

    plt.title(f"Error Distribution Analysis | Subnet {sid_str}", fontsize=14, fontweight='bold')
    plt.xlabel("Absolute Reconstruction Error (Log Scale)", fontsize=12)
    plt.ylabel("Frequency (Count)", fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)


    save_path = ANOMALY_DIR / f"distribution_subnet_{sid_str}.png"
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()
    plt.close()

    print(f" Saved Distribution Plot to: {save_path}")

print("\nAll plots generated.")

In [ ]:
from darts.models import RNNModel
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- CONFIG ---
MODEL_PATH = Path("/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_log/gru_168_1layer_subnets_log.pth.tar")
ANOMALY_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Error Distribution")
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)

K_SIGMA = 3.0
WINDOW_SIZE = 24

# Set a professional plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150  # High resolution

# --- LOAD MODEL ---
print(f"Loading Champion Model...")
model = RNNModel.load(str(MODEL_PATH))

# Pick your target subnets
targets = valid_subnet_ids[:10]
print(f"Generating professional plots for {len(targets)} subnets...")

for sid in targets:
    sid_str = str(sid)

    # 1. Get Data
    if sid_str not in train_scaled_log: continue

    train_ts = train_scaled_log[sid_str]
    val_ts   = val_scaled_log[sid_str]
    test_ts  = test_scaled_log[sid_str]
    full_series = train_ts.concatenate(val_ts).concatenate(test_ts)

    # 2. Predict
    pred = model.historical_forecasts(
        series=full_series,
        start=test_ts.start_time(),
        forecast_horizon=1,
        stride=1,
        retrain=False,
        verbose=False,
        last_points_only=True
    )

    # 3. Calculate Error
    intersect_idx = test_ts.time_index.intersection(pred.time_index)
    actual_aligned = test_ts.slice_intersect(pred)
    pred_aligned = pred.slice_intersect(test_ts)

    # Error Series
    error_series = (actual_aligned - pred_aligned).map(np.abs)
    error_values = error_series.values().flatten()

    # 4. Dynamic Threshold
    error_pd = pd.Series(error_values, index=error_series.time_index)
    rolling_mean = error_pd.rolling(window=WINDOW_SIZE, min_periods=1).mean()
    rolling_std  = error_pd.rolling(window=WINDOW_SIZE, min_periods=1).std()
    threshold = rolling_mean + (K_SIGMA * rolling_std)


In [ ]:
# --- 6. SUMMARY STATISTICS ---
if len(anomalies) > 0:
    anomaly_rate = (len(anomalies) / len(test_ts)) * 100
    print(f"\n--- Analysis for Subnet {sid_str} ---")
    print(f"Total Data Points: {len(test_ts)}")
    print(f"Anomalies Detected: {len(anomalies)}")
    print(f"Anomaly Rate: {anomaly_rate:.2f}%")

    if anomaly_rate > 5.0:
        print("High anomaly rate! Consider increasing K_SIGMA to 4.0")
    else:
        print("Anomaly rate is within normal range (typically 1-5%).")

In [ ]:
from darts.models import RNNModel
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- CONFIG ---
MODEL_PATH = Path("/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_log/gru_168_1layer_subnets_log.pth.tar")
SAVE_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Model_Analysis_Plots")
SAVE_DIR.mkdir(parents=True, exist_ok=True)


plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150

# --- LOAD MODEL ---
print("Loading Champion Model (GRU 1-Layer)...")
model = RNNModel.load(str(MODEL_PATH))


targets = ['0', '9', '258']

print(f"Generating Model Analysis plots for: {targets}")

for sid in targets:
    sid_str = str(sid)
    if sid_str not in train_scaled_log: continue

    # 1. Get Data
    train = train_scaled_log[sid_str]
    val   = val_scaled_log[sid_str]
    test  = test_scaled_log[sid_str]

    # Combined series for context
    full_series = train.concatenate(val).concatenate(test)

    # 2. Predict (Test Set Only)
    pred = model.historical_forecasts(
        series=full_series,
        start=test.start_time(),
        forecast_horizon=1,
        stride=1,
        retrain=False,
        verbose=False,
        last_points_only=True
    )

    # 3. PLOT: FORECAST VS ACTUAL
    plt.figure(figsize=(12, 5))

    # Plot Actual
    test.plot(label="Actual Traffic (Log)", color='#2c3e50', alpha=0.7, linewidth=1.5)

    # Plot Prediction
    pred.plot(label="GRU Forecast", color='#e67e22', alpha=0.9, linewidth=1.5)

    plt.title(f"Model Fit Analysis | Subnet {sid_str} | GRU 1-Layer", fontsize=14, fontweight='bold')
    plt.ylabel("Log Volume", fontsize=12)
    plt.xlabel("Date", fontsize=12)
    plt.legend(frameon=True, framealpha=0.9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    # Save
    plt.savefig(SAVE_DIR / f"model_fit_subnet_{sid_str}.png")
    plt.close()
    print(f"Saved analysis plot for Subnet {sid_str}")

In [ ]:
from darts.models import NaiveSeasonal
from darts.metrics import rmse, r2_score
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --- CONFIG ---
SAVE_DIR = Path("/content/drive/MyDrive/Thesis/New_Results/Baselines")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# --- LOAD DATA (Existing dictionaries)---
print(f"Evaluated Naive Baseline on {len(valid_subnet_ids)} subnets...")

baseline_results = []

# We use K=168 (1 week)
naive_model = NaiveSeasonal(K=168)

for sid in valid_subnet_ids:
    sid_str = str(sid)

    # Get Data
    train = train_scaled_log[sid_str]
    val   = val_scaled_log[sid_str]
    test  = test_scaled_log[sid_str]

    # We fit on train+val
    train_val = train.concatenate(val)

    try:
        # Predict the entire test range
        naive_model.fit(train_val)
        pred = naive_model.predict(len(test))

        # Metrics
        r2 = r2_score(test, pred)
        err = rmse(test, pred)

        baseline_results.append({
            "Subnet": sid_str,
            "Naive_R2": r2,
            "Naive_RMSE": err
        })

    except Exception as e:
        print(f"Error on {sid_str}: {e}")

# --- SUMMARY ---
df_base = pd.DataFrame(baseline_results)
print("\n== NAIVE BASELINE RESULTS ===")
print(f"Mean R2:   {df_base['Naive_R2'].mean():.4f}")
print(f"Mean RMSE: {df_base['Naive_RMSE'].mean():.4f}")

# Compare with GRU
print(f"vs GRU (Log): ~0.5637 (R2) / ~0.1026 (RMSE)")

df_base.to_csv(SAVE_DIR / "naive_baseline_metrics.csv", index=False)

In [ ]:
#import time
#import pandas as pd
#from darts.models import RNNModel
#from pathlib import Path
#
## --- CONFIG ---
#MODEL_PATH = Path("/content/drive/MyDrive/Thesis/New_Results/Subnets/GRU_1layer_log/gru_168_1layer_subnets_log.pth.tar")
#
## Load Model
#print("Loading model for speed test...")
#model = RNNModel.load(str(MODEL_PATH))
#
#
#test_series = test_scaled_log['9']
#input_chunk = test_series[-168:] # Simulating the last week of data input
#
#
#print("Running latency test...")
#start_time = time.time()
#iterations = 100
#
#for _ in range(iterations):
#    # Predict 1 step into the future (Real-world scenario)
#    _ = model.predict(n=1, series=input_chunk, verbose=False)
#
#end_time = time.time()
#
## --- CALCULATIONS ---
#total_time = end_time - start_time
#avg_time_per_subnet = total_time / iterations
#subnets_in_isp = 10000
#
#print(f"\n=== SYSTEM LATENCY RESULTS ===")
#print(f"Average Inference Time per Subnet: {avg_time_per_subnet:.4f} seconds ({avg_time_per_subnet*1000:.2f} ms)")
#print(f"Projected Time to Scan {subnets_in_isp} Subnets: {avg_time_per_subnet * subnets_in_isp:.2f} seconds")
#
#if (avg_time_per_subnet * subnets_in_isp) < 300: # Less than 5 minutes
#    print("VERDICT: REAL-TIME CAPABLE (Can scan entire network in < 5 mins)")
#else:
#    print("VERDICT: OPTIMIZATION NEEDED")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- CONFIG ---
DATA_DIR   = Path("/content/drive/MyDrive/Thesis/Datasets/network_anomaly")
SUBNET_DIR = DATA_DIR / "institution_subnets/agg_1_hour"
TIMES_PATH = DATA_DIR / "times/times_1_hour.csv"
TARGET_SID = '9'  # The subnet we want to investigate

# 1. Load the Time Mapping
print("Loading time mapping...")
times_df = pd.read_csv(TIMES_PATH)
times_df["time"] = pd.to_datetime(times_df["time"], errors="coerce")
if times_df["time"].dt.tz is not None:
    times_df["time"] = times_df["time"].dt.tz_localize(None)
times_df["id_time"] = times_df["id_time"].astype(int)

# 2. Load ONLY Subnet 9 (Raw, with ALL columns)
print(f"Loading raw data for Subnet {TARGET_SID}...")
file_path = list(SUBNET_DIR.glob(f"*{TARGET_SID}.csv"))[0]
df = pd.read_csv(file_path)

# 3. Merge to get real dates
df["id_time"] = df["id_time"].astype(int)
df = df.merge(times_df, on="id_time", how="left")
df = df.sort_values("time").set_index("time")

# 4. Select the "Case Study" Window
n_points = len(df)
subset = df.iloc[int(n_points * 0.75):].copy()

# --- PLOT ---
print("Generating Forensic Plot...")
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Plot A: Bytes (Volume) - The Detection
axes[0].plot(subset.index, subset['n_bytes'], color='#2c3e50', linewidth=1)
axes[0].set_title(f"1. Detection Signal: Volume (Subnet {TARGET_SID})", fontsize=12, fontweight='bold', loc='left')
axes[0].set_ylabel("Bytes")
axes[0].grid(True, alpha=0.3)

# Plot B: Flows (Connections) - The Context
axes[1].plot(subset.index, subset['n_flows'], color='#e67e22', linewidth=1)
axes[1].set_title("2. Context Signal: Flow Count", fontsize=12, fontweight='bold', loc='left')
axes[1].set_ylabel("Flows")
axes[1].grid(True, alpha=0.3)

# Plot C: Protocol Ratio (Type) - The Details
# Handle potential missing values or infinity in ratio
subset['tcp_udp_ratio_packets'] = subset['tcp_udp_ratio_packets'].fillna(0)
axes[2].plot(subset.index, subset['tcp_udp_ratio_packets'], color='#27ae60', linewidth=1)
axes[2].set_title("3. Type Signal: TCP/UDP Ratio", fontsize=12, fontweight='bold', loc='left')
axes[2].set_ylabel("Ratio")
# Clip y-axis if ratio explodes (e.g., 1000:1) to keep plot readable
axes[2].set_ylim(0, subset['tcp_udp_ratio_packets'].quantile(0.99) * 1.5)
axes[2].grid(True, alpha=0.3)

plt.xlabel("Date", fontsize=12)
plt.tight_layout()
plt.show()